## The Xtrack Environment
For more info see [Environment Section in the User's guide](https://xsuite.readthedocs.io/en/latest/environment.html)

An xtrack **Environment** is the shared container that keeps together:

 - **Variables** in env.vars: scalar knobs and deferred expressions used to drive elements and lines.
 - **Elements** in env.elements: magnets, markers, RF cavities, etc. They can be reused across multiple lines.
 - **Lines** in env.lines: ordered sequences of elements assembled from the environment.
 - **Additional data** including reference particles and user-defined functions.

In [52]:
import xtrack as xt

### Create an environment

In [53]:
# Create an environment
env = xt.Environment()

In [54]:
env

Environment(0 lines: {}, 0 elements, 1 vars, 0 particles)

### Define variables

In [55]:
# Define variables
env['t_turn_s'] = 0.0
env['l_q'] = 1.0
env['kq'] = 0.12
env['kq.trim'] = '0.05 * kq' # deferred expression (is updated when kq changes)
env['kq.total'] = 'kq + kq.trim' # deferred expression

In [56]:
# Access variables
env['kq'], env['kq.trim'], env['kq.total']

(0.12, 0.006, 0.126)

In [57]:
# Behavior of deferred expressions
env['kq'] = 0.06
env['kq'], env['kq.trim'], env['kq.total']

(0.06, 0.003, 0.063)

### Define elements

In [58]:
# Create elements using  variables defined above
env.new('qf', xt.Quadrupole, length='l_q', k1='kq.total')
env.new('qd', xt.Quadrupole, length='l_q', k1='-kq.total')
env.new('dr', xt.Drift, length=6.0);

In [59]:
# Deferred expressions act on the elements
env['qf'].k1

np.float64(0.063)

In [60]:
env['kq'] = 0.12
env['qf'].k1

np.float64(0.126)

### Define beam lines

In [61]:
# Define two lines using the elements above
env.new_line(name='fodo', components=['qf', 'dr', 'qd', 'dr'])
env.new_line(name='half_fodo', components=['qf', 'dr']);

### Inspect environment containers

In [62]:
env

Environment(2 lines: {fodo, half_fodo}, 3 elements, 5 vars, 0 particles)

In [65]:
# Inspect variables
env.vars

EnvVars(5 vars: {t_turn_s, l_q, kq, kq.trim, kq.total, ...})

In [66]:
env.vars.get_table()

VarsTable: 5 rows, 3 cols
name             value expr          
t_turn_s             0 None          
l_q                  1 None          
kq                0.12 None          
kq.trim          0.006 (0.05 * kq)   
kq.total         0.126 (kq + kq.trim)

In [68]:
# Inspect elements
env.elements.get_table()

LineTable: 3 rows, 9 cols
name element_type isthick isreplica parent_name parent_type prototype iscollective        length
dr   Drift           True     False None        None        None             False             6
qd   Quadrupole      True     False None        None        None             False             1
qf   Quadrupole      True     False None        None        None             False             1

In [70]:
# Inspect lines
env.lines.get_table()

Table: 2 rows, 3 cols
name      num_elements mode  
fodo                 4 normal
half_fodo            2 normal